In [1]:
import torch
from torch import nn
from torch.nn import functional as f
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split

from torchmetrics.classification import Accuracy, Recall


In [ ]:
import random
import os
import numpy as np
import torch

def set_seed(seed=42):
 
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    np.random.seed(seed)        

set_seed(616) 
print("ok")

ok


In [4]:
df = pd.read_csv("/kaggle/input/datasets/fatemehmehrparvar/liver-disorders/Indian Liver Patient Dataset (ILPD).csv")
df.head(10)

df['A/G Ratio'] = df['A/G Ratio'].fillna(df['A/G Ratio'].mean())

labelencoder = LabelEncoder()
onehotencoder = OneHotEncoder()

df["Gender"] = labelencoder.fit_transform(df["Gender"])

onehot_df = onehotencoder.fit_transform(df[["Gender"]])
onehot_df = pd.DataFrame(onehot_df.toarray(), columns=["gender_male", "gender_female"])
df = pd.concat([df, onehot_df], axis=1)
df = df.drop(columns='Gender')

normal_liver = df[df["Selector"] == 2]
normal_liver = normal_liver.drop(columns = "Selector")

real_1 = df[df["Selector"] == 1]
real_1 = real_1.drop(columns = "Selector")

y = df["Selector"]
x = df.drop(columns = "Selector")


s_scaler = StandardScaler()
x = s_scaler.fit_transform(x)
normal_liver = s_scaler.fit_transform(normal_liver)
real_1 = s_scaler.fit_transform(real_1)

m_scaler = MinMaxScaler()
x = m_scaler.fit_transform(x)
normal_liver = m_scaler.fit_transform(normal_liver)
real_1 = m_scaler.fit_transform(real_1)

print("ok")

ok


In [5]:
x = x.astype(np.float32)
normal_liver = normal_liver.astype(np.float32)
real_1 = real_1 = real_1.astype(np.float32)

y = y.to_numpy()
y = y.astype(np.float32)

x = torch.from_numpy(x)
y = torch.from_numpy(y)
normal_liver = torch.from_numpy(normal_liver)
real_1 = torch.from_numpy(real_1)

y = (y == 1).float()
y.sum().item()
print("ok")

ok


In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=423)

In [7]:
class VAE(nn.Module):

    def __init__(self, input_dim=11, hidden_dim=32, latent_dim=8):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = f.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-20.0, max=20.0)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = f.relu(self.fc2(z))
        return f.sigmoid(self.fc3(h)) 

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)
        return reconstruction, mu, logvar

def vae_loss(recon_x, x, mu, logvar, kl_weight=0.01, beta = 20.0):
    recon_loss = f.binary_cross_entropy(recon_x, x, reduction="mean")
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    total_loss = recon_loss + beta * (kl_weight * kl_loss)
    return total_loss, recon_loss, kl_loss

vae_dataset = TensorDataset(normal_liver)
vae_loader = DataLoader(vae_dataset, batch_size=64, shuffle=True)

VAE_model = VAE()
optimizer = Adam(VAE_model.parameters(), lr=0.001)

epochs = 400

for epoch in range(epochs):
    VAE_model.train()
    total_VAEloss = 0
    total_recon_loss = 0
    total_kl_loss = 0
    
    kl_weight = min(1.0, epoch / 50.0) * 0.1 

    for batch in vae_loader:
        x = batch[0]
        
        optimizer.zero_grad()
        recon, mu, logvar = VAE_model(x)
        VAEloss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, kl_weight, beta = 20.0)

        VAEloss.backward()
        optimizer.step()
        
        total_VAEloss += VAEloss.item() * len(x)
        total_recon_loss += recon_loss.item() * len(x)
        total_kl_loss += kl_loss.item() * len(x)

    total_samples = len(normal_liver)
    avg_loss = total_VAEloss / total_samples
    avg_recon_loss = total_recon_loss / total_samples
    avg_kl_loss = total_kl_loss / total_samples

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.4f} | "
            f"Recon: {avg_recon_loss:.4f} | "
            f"KL: {avg_kl_loss:.4f} (wt: {kl_weight:.3f})"
        )


Epoch [1/400] Loss: 0.7256 | Recon: 0.7256 | KL: 0.0279 (wt: 0.000)
Epoch [10/400] Loss: 0.6658 | Recon: 0.6579 | KL: 0.0221 (wt: 0.018)
Epoch [20/400] Loss: 0.6191 | Recon: 0.6132 | KL: 0.0078 (wt: 0.038)
Epoch [30/400] Loss: 0.5788 | Recon: 0.5745 | KL: 0.0037 (wt: 0.058)
Epoch [40/400] Loss: 0.5504 | Recon: 0.5471 | KL: 0.0021 (wt: 0.078)
Epoch [50/400] Loss: 0.5344 | Recon: 0.5320 | KL: 0.0012 (wt: 0.098)
Epoch [60/400] Loss: 0.5271 | Recon: 0.5254 | KL: 0.0009 (wt: 0.100)
Epoch [70/400] Loss: 0.5206 | Recon: 0.5191 | KL: 0.0008 (wt: 0.100)
Epoch [80/400] Loss: 0.5175 | Recon: 0.5161 | KL: 0.0007 (wt: 0.100)
Epoch [90/400] Loss: 0.5184 | Recon: 0.5171 | KL: 0.0006 (wt: 0.100)
Epoch [100/400] Loss: 0.5130 | Recon: 0.5118 | KL: 0.0006 (wt: 0.100)
Epoch [110/400] Loss: 0.5160 | Recon: 0.5148 | KL: 0.0006 (wt: 0.100)
Epoch [120/400] Loss: 0.5189 | Recon: 0.5177 | KL: 0.0006 (wt: 0.100)
Epoch [130/400] Loss: 0.5141 | Recon: 0.5131 | KL: 0.0005 (wt: 0.100)
Epoch [140/400] Loss: 0.5132 | 

In [7]:
batch_size = 80
latent_dim = 8 

tensor = torch.randn(batch_size, latent_dim)
y_tensor = torch.zeros(batch_size)

new_data = VAE_model.decode(tensor).detach()

x_train_with_VAE = torch.cat((x_train, new_data), dim=0)
y_train_with_VAE = torch.cat((y_train, y_tensor), dim=0)

perm_with_vae = torch.randperm(x_train_with_VAE.size(0))
x_train_with_VAE = x_train_with_VAE[perm_with_vae]
y_train_with_VAE = y_train_with_VAE[perm_with_vae]

perm_original = torch.randperm(x_train.size(0))
x_train = x_train[perm_original]
y_train = y_train[perm_original]

print("ok")
print(len(y_train_with_VAE) - y_train_with_VAE.sum())



ok
tensor(217.)


In [ ]:

VAE_model.eval()
with torch.no_grad():

    mu, logvar = VAE_model.encode(x_train) 

    recon_x = VAE_model.decode(mu)


recon_error = torch.abs(x_train - recon_x)


x_train_with_VAE = torch.cat((x_train, recon_error), dim=1)
y_train_with_VAE = y_train.clone() 

perm_with_vae = torch.randperm(x_train_with_VAE.size(0))
x_train_with_VAE = x_train_with_VAE[perm_with_vae]
y_train_with_VAE = y_train_with_VAE[perm_with_vae]


新特徵矩陣維度: torch.Size([466, 22])
總正常樣本數 (Label=0): 137.0


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE


X_real_0 = normal_liver
X_real_1 = real_1
X_vae_0 = new_data
X = np.concatenate([X_real_0, X_real_1, X_vae_0], axis=0)


y = np.concatenate(
    [
        np.zeros(len(X_real_0)),
        np.ones(len(X_real_1)),
        np.ones(len(X_vae_0)) * 2,
    ]
)


tsne = TSNE(n_components=2, perplexity=55, n_iter=1000, random_state=42)
X_embedded = tsne.fit_transform(X)


plt.figure(figsize=(10, 8))
colors = ["blue", "red", "green"]
labels = ["Real 0", "Real 1", "VAE Generated 0"]

for i in range(3):
    idx = y == i
    plt.scatter(
        X_embedded[idx, 0],
        X_embedded[idx, 1],
        c=colors[i],
        label=labels[i],
        alpha=0.6,
    )

plt.legend()
plt.title("t-SNE Diagnostic Plot")
plt.show()

NameError: name 'new_data' is not defined

In [26]:
class ResBlock(nn.Module):
    def __init__(self, features = 512, bottleneck_features = 64):
        super().__init__()

        self.bn1 = nn.BatchNorm1d(features)
        self.fnc1 = nn.Linear(features, bottleneck_features)
        
        self.bn2 = nn.BatchNorm1d(bottleneck_features)
        self.fnc2 = nn.Linear(bottleneck_features, bottleneck_features)

        self.bn3 = nn.BatchNorm1d(bottleneck_features)
        self.fnc3 = nn.Linear(bottleneck_features, features)
        
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = f.relu(self.bn1(x))
        out = self.dropout(self.fnc1(out))
        
        out = f.relu(self.bn2(out))
        out = self.fnc2(out)
        
        out = f.relu(self.bn3(out))
        out = self.fnc3(out)
        
        return x + out

class BranchBlock(nn.Module):
    def __init__(self, features = 512, depth = 3):
        super().__init__()
        self.res_layers = nn.ModuleList([
            ResBlock(features = features) for _ in range(depth)
        ])

    def forward(self, x):
        for layer in self.res_layers:
            x = layer(x)
        return x
class ElementWeight(nn.Module):
    def __init__(self, features = 512):
        super().__init__()
        self.fnc1 = nn.Linear(features, features)

    def forward(self, x):
        weights = torch.sigmoid(self.fnc1(x))

        return x * weights

class Detection_model(nn.Module):
    def __init__(self, num_resBlocks = 5, num_branchs = 5):
        super().__init__()
        self.input = nn.Linear(22, 512)
        self.bn_in = nn.BatchNorm1d(512)
        
        self.fnc1 = nn.Linear(512, 512)
        self.bn512_1 = nn.BatchNorm1d(512)

        self.parallel_branchs = nn.ModuleList([
            BranchBlock(features = 512, depth = num_resBlocks) for _ in range(num_branchs)
        ])

        self.weights = ElementWeight(features = 512)
                
        self.fnc2 = nn.Linear(512, 512)
        self.bn512_2 = nn.BatchNorm1d(512)
        self.output = nn.Linear(512, 1)

    def forward(self, x):
        x = f.relu(self.bn_in(self.input(x)))
        x = f.relu(self.bn512_1(self.fnc1(x)))

        parallel = sum(blocks(x) for blocks in self.parallel_branchs)
        parallel = self.weights(parallel)
        x = x + parallel 

        x = f.relu(self.bn512_2(self.fnc2(x)))
        x = self.output(x)

        return x

    def prediction(self, x):
        return self.forward(x)

print("ok")

ok


In [27]:
mod = 1

if mod == 0:
    training_data = x_train
    target = y_train
else:
    training_data = x_train_with_VAE
    target = y_train_with_VAE

train_dataset = TensorDataset(training_data, target)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

detection_model = Detection_model(num_resBlocks = 5, num_branchs = 5)
optim = AdamW(detection_model.parameters(), lr = 0.0001)

scheduler = ReduceLROnPlateau(optim, mode='min', factor=0.3, patience=10)

accuracy = Accuracy(task = "binary")
recall = Recall(task="binary")

epochs = 150

pos_weight = torch.tensor([0.8])

for epoch in range(epochs):
    detection_model.train()
    total_loss = 0
    
    epoch_preds = []
    epoch_targets = []

    for batch_x, batch_y in train_loader:
        optim.zero_grad()
        
        y_pred_logits = detection_model(batch_x)
        
        y_pred_logits = y_pred_logits.squeeze() 
        batch_y = batch_y.squeeze()

        loss = f.binary_cross_entropy_with_logits(y_pred_logits, batch_y, pos_weight = pos_weight)
        loss.backward()
        optim.step()

        total_loss += loss.item() * len(batch_y)
        
        y_pred_prob = torch.sigmoid(y_pred_logits)
        epoch_preds.append(y_pred_prob.detach())
        epoch_targets.append(batch_y.detach())
        
    avg_loss = total_loss / len(training_data)

    torch.nn.utils.clip_grad_norm_(detection_model.parameters(), max_norm=1.0)
    scheduler.step(avg_loss)
    
    all_preds = torch.cat(epoch_preds)
    all_targets = torch.cat(epoch_targets)
    
    acc_score = accuracy(all_preds, all_targets)
    recall_score = recall(all_preds, all_targets)
    
    if (epoch+1) % 10 == 0:
        
        current_lr = optim.param_groups[0]['lr']
        print(f"epoch:[{epoch+1}/{epochs}]  loss: {avg_loss:.4f}  acc: {acc_score:.4f}  recall:{recall_score:.4f}  lr:{current_lr:.10f}") 


epoch:[10/150]  loss: 0.3488  acc: 0.8155  recall:0.8480  lr:0.0001000000
epoch:[20/150]  loss: 0.2858  acc: 0.8584  recall:0.8906  lr:0.0001000000
epoch:[30/150]  loss: 0.2226  acc: 0.8884  recall:0.9058  lr:0.0001000000
epoch:[40/150]  loss: 0.2071  acc: 0.9099  recall:0.9362  lr:0.0001000000
epoch:[50/150]  loss: 0.1740  acc: 0.9056  recall:0.9210  lr:0.0001000000
epoch:[60/150]  loss: 0.1680  acc: 0.9249  recall:0.9483  lr:0.0001000000
epoch:[70/150]  loss: 0.1546  acc: 0.9227  recall:0.9331  lr:0.0000300000
epoch:[80/150]  loss: 0.1351  acc: 0.9378  recall:0.9544  lr:0.0000300000
epoch:[90/150]  loss: 0.0989  acc: 0.9592  recall:0.9818  lr:0.0000300000
epoch:[100/150]  loss: 0.1116  acc: 0.9528  recall:0.9696  lr:0.0000090000
epoch:[110/150]  loss: 0.0885  acc: 0.9635  recall:0.9757  lr:0.0000027000
epoch:[120/150]  loss: 0.0938  acc: 0.9742  recall:0.9818  lr:0.0000008100
epoch:[130/150]  loss: 0.1074  acc: 0.9592  recall:0.9696  lr:0.0000008100
epoch:[140/150]  loss: 0.0981  acc

In [ ]:
import torch


VAE_model.eval()

with torch.no_grad():
    mu, logvar = VAE_model.encode(x_test) 
    recon_x = VAE_model.decode(mu)


recon_error = torch.abs(x_test - recon_x)

x_test_with_VAE = torch.cat((x_test, recon_error), dim=1) 
y_test_with_VAE = y_train.clone()

perm_with_vae = torch.randperm(x_test_with_VAE.size(0))
x_test_with_VAE = x_test_with_VAE[perm_with_vae]
y_test_with_VAE = y_test_with_VAE[perm_with_vae]





In [ ]:
dataset = TensorDataset(x_test_with_VAE, y_test)
loader = DataLoader(dataset, batch_size=16, shuffle=False) 

all_preds = []
all_targets = []

detection_model.eval()

with torch.no_grad():
    for batch_x, batch_y in loader:
        
        y_logits = detection_model(batch_x)
        prob = torch.sigmoid(y_logits)
        
        y_pred = (prob >= 0.5).int()
        
        all_preds.append(y_pred.view(-1))
        all_targets.append(batch_y.view(-1))

final_preds = torch.cat(all_preds, dim=0)
final_targets = torch.cat(all_targets, dim=0).int()

accuracy.reset()
recall.reset()
metric_acc = accuracy(final_preds, final_targets)
metric_recall = recall(final_preds, final_targets)

print(f"全測試集評估結果")
print(f"Accuracy: {acc_score.item():.4f}")
print(f"Recall:   {recall_score.item():.4f}")

print("\n預測的前 50 筆結果:")
print(final_preds[:50].cpu().numpy()) 

--- 全測試集評估結果 ---
Accuracy: 0.9657
Recall:   0.9757

預測的前 50 筆結果:
[1 0 1 1 0 1 1 1 1 0 1 1 1 0 1 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 0 1 1 1 1 1 1 1 1 0 1 1 0]
